In [1]:
# Qwen2.5-3B zero-shot 평가 먼저
import torch, os, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.cuda.empty_cache()
gc.collect()

MODEL_NAME_3B = "Qwen/Qwen2.5-3B-Instruct"
print(f"모델 로드 중: {MODEL_NAME_3B}")

tokenizer_3b = AutoTokenizer.from_pretrained(MODEL_NAME_3B)
model_3b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_3B,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_3b.eval()
print("로드 완료!")
print(f"GPU 메모리: {torch.cuda.memory_reserved(0)/1024**3:.1f} GB")

/opt/jupyter/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델 로드 중: Qwen/Qwen2.5-3B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 232.25it/s]

로드 완료!
GPU 메모리: 5.8 GB


In [3]:
import os, json, torch, gc
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import f1_score

CONDITIONS   = ["reply_only", "useful", "irrelevant", "conflicting", "mixed", "lexical"]
LABEL_MAP    = {"support": 0, "deny": 1, "query": 2, "comment": 3}
VALID_LABELS = {"support", "deny", "query", "comment"}

with open('../context_conditions.json', encoding='utf-8') as f:
    full_dataset = json.load(f)
dataset    = [d for d in full_dataset if d['split'] == 'dev']
train_data = [d for d in full_dataset if d['split'] == 'train' and d['useful'] is not None]
print(f"dev: {len(dataset)}개 / train: {len(train_data)}개")

SYSTEM_PROMPT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""

def build_prompt(text: str) -> list:
    cleaned = text
    cleaned = cleaned.replace("[Source]",     "Rumour post:")
    cleaned = cleaned.replace("[Context]",    "Previous reply:")
    cleaned = cleaned.replace("[Misleading]", "Another reply:")
    cleaned = cleaned.replace("[Target]",     "Reply to classify:")
    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{cleaned}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

def evaluate(result: dict) -> dict:
    golds = result["golds"]
    preds = result["preds"]
    if len(golds) == 0:
        return {"macro_f1": 0.0, "per_class": [0.0] * 4}
    macro_f1  = f1_score(golds, preds, average="macro", zero_division=0)
    per_class = f1_score(golds, preds, average=None, labels=[0,1,2,3], zero_division=0)
    return {"macro_f1": macro_f1, "per_class": per_class.tolist()}

def run_condition(condition: str, data: list, predict_fn) -> dict:
    golds, preds  = [], []
    invalid_count = 0
    skipped_count = 0
    total         = len(data)
    for i, d in enumerate(data):
        if d[condition] is None:
            skipped_count += 1
            continue
        pred = predict_fn(d[condition])
        if pred == "invalid":
            invalid_count += 1
            continue
        golds.append(LABEL_MAP[d["label"]])
        preds.append(LABEL_MAP[pred])
        if (i + 1) % 100 == 0:
            print(f"  [{condition}] {i+1}/{total} | 유효 {len(golds)} | invalid {invalid_count}")
    return {
        "golds":         golds,
        "preds":         preds,
        "invalid_count": invalid_count,
        "skipped_count": skipped_count,
        "n":             len(golds),
    }

def make_predict_fn(model, tokenizer):
    def predict(text: str, max_new_tokens: int = 10) -> str:
        messages   = build_prompt(text)
        text_input = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        input_ids = tokenizer(text_input, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output_ids[0][input_ids.shape[-1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        for label in VALID_LABELS:
            if label in raw_output:
                return label
        return "invalid"
    return predict

print("함수 정의 완료!")

dev: 1447개 / train: 4890개
함수 정의 완료!


In [4]:
# 3B predict 함수 + zero-shot 평가
predict_3b = make_predict_fn(model_3b, tokenizer_3b)

all_results_3b = {}
for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")
    result  = run_condition(condition, dataset, predict_fn=predict_3b)
    metrics = evaluate(result)
    all_results_3b[condition] = {"raw": result, "metrics": metrics}
    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")

# 결과 저장
import json
save_data_3b = {}
for condition, data in all_results_3b.items():
    save_data_3b[condition] = {
        "golds":         data["raw"]["golds"],
        "preds":         data["raw"]["preds"],
        "invalid_count": data["raw"]["invalid_count"],
        "n":             data["raw"]["n"],
        "macro_f1":      data["metrics"]["macro_f1"],
        "per_class_f1":  data["metrics"]["per_class"],
    }
with open('experiment_results_3B_zeroshot.json', 'w', encoding='utf-8') as f:
    json.dump(save_data_3b, f, ensure_ascii=False, indent=2)
print("\n저장 완료: experiment_results_3B_zeroshot.json")

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



  조건: reply_only
  [reply_only] 100/1447 | 유효 100 | invalid 0
  [reply_only] 200/1447 | 유효 200 | invalid 0
  [reply_only] 300/1447 | 유효 300 | invalid 0
  [reply_only] 400/1447 | 유효 400 | invalid 0
  [reply_only] 500/1447 | 유효 500 | invalid 0
  [reply_only] 600/1447 | 유효 600 | invalid 0
  [reply_only] 700/1447 | 유효 700 | invalid 0
  [reply_only] 800/1447 | 유효 800 | invalid 0
  [reply_only] 900/1447 | 유효 900 | invalid 0
  [reply_only] 1000/1447 | 유효 1000 | invalid 0
  [reply_only] 1100/1447 | 유효 1100 | invalid 0
  [reply_only] 1200/1447 | 유효 1200 | invalid 0
  [reply_only] 1300/1447 | 유효 1300 | invalid 0
  [reply_only] 1400/1447 | 유효 1400 | invalid 0
  완료 → n=1447 | invalid=0
  macro-F1 : 0.3448
  support=0.026  deny=0.163  query=0.416  comment=0.775

  조건: useful
  [useful] 100/1447 | 유효 100 | invalid 0
  [useful] 200/1447 | 유효 200 | invalid 0
  [useful] 300/1447 | 유효 300 | invalid 0
  [useful] 400/1447 | 유효 400 | invalid 0
  [useful] 500/1447 | 유효 500 | invalid 0
  [useful] 600/1447 |

In [2]:
import os, gc, torch, json
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from collections import Counter

# 데이터 로드
with open('../context_conditions.json', encoding='utf-8') as f:
    full_dataset = json.load(f)
train_data = [d for d in full_dataset if d['split'] == 'train' and d['useful'] is not None]
print(f"train 샘플 수: {len(train_data)}")

SYSTEM_PROMPT_FT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""

def format_sample_3b(d, condition='useful'):
    text = d[condition]
    text = text.replace("[Source]",     "Rumour post:")
    text = text.replace("[Context]",    "Previous reply:")
    text = text.replace("[Misleading]", "Another reply:")
    text = text.replace("[Target]",     "Reply to classify:")
    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{text}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return {
        "text": (
            f"<|im_start|>system\n{SYSTEM_PROMPT_FT}<|im_end|>\n"
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n{d['label']}<|im_end|>"
        )
    }

# adversarial 데이터 구성
adv_3b = []
for d in train_data:
    adv_3b.append(format_sample_3b(d, 'useful'))
    if d.get('conflicting') is not None:
        adv_3b.append(format_sample_3b(d, 'conflicting'))
    if d.get('mixed') is not None:
        adv_3b.append(format_sample_3b(d, 'mixed'))

adv_dataset_3b = Dataset.from_list(adv_3b)
print(f"학습 데이터: {len(adv_dataset_3b)}개")

# checkpoint에서 이어서 학습
checkpoint_path = os.path.expanduser("~/qwen_3b_adv/checkpoint-805")
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer_3b = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer_3b.pad_token = tokenizer_3b.eos_token

model_3b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model_3b = get_peft_model(model_3b, lora_config)
model_3b.print_trainable_parameters()

sft_config = SFTConfig(
    output_dir=os.path.expanduser("~/qwen_3b_adv"),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model_3b,
    args=sft_config,
    train_dataset=adv_dataset_3b,
    processing_class=tokenizer_3b,
)

print("checkpoint에서 이어서 학습 시작...")
trainer.train(resume_from_checkpoint=checkpoint_path)
print("완료!")

save_path = os.path.expanduser("~/qwen_3b_adv/final")
model_3b.save_pretrained(save_path)
tokenizer_3b.save_pretrained(save_path)
print(f"저장 완료: {save_path}")

train 샘플 수: 4890
학습 데이터: 12878개


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 230.51it/s]


trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


Tokenizing train dataset: 100%|██████████| 12878/12878 [00:06<00:00, 1884.77 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


checkpoint에서 이어서 학습 시작...


Step,Training Loss
850,0.809673
900,0.780174
950,0.736492
1000,0.738349
1050,0.731892
1100,0.713775
1150,0.696608
1200,0.676109
1250,0.657805
1300,0.654051


완료!
저장 완료: /home/ubuntu/qwen_3b_adv/final


In [3]:
import os, json, torch, gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import f1_score
from collections import Counter

# 공통 변수 정의
CONDITIONS   = ["reply_only", "useful", "irrelevant", "conflicting", "mixed", "lexical"]
LABEL_MAP    = {"support": 0, "deny": 1, "query": 2, "comment": 3}
VALID_LABELS = {"support", "deny", "query", "comment"}

with open('../context_conditions.json', encoding='utf-8') as f:
    full_dataset = json.load(f)
dataset = [d for d in full_dataset if d['split'] == 'dev']
print(f"dev 샘플 수: {len(dataset)}")

SYSTEM_PROMPT = """You are a stance classification expert for social media discussions about rumours.

Classify the stance of the TARGET reply using exactly one of these four labels:

- support: The reply explicitly states the rumour IS true or confirmed.
- deny: The reply explicitly states the rumour IS false or fabricated.
- query: The reply asks for sources, evidence, or verification.
- comment: Everything else. The reply does not directly address whether the rumour is true or false.

Respond with ONLY one word: support, deny, query, or comment. No explanation."""

def build_prompt(text):
    cleaned = text
    cleaned = cleaned.replace("[Source]",     "Rumour post:")
    cleaned = cleaned.replace("[Context]",    "Previous reply:")
    cleaned = cleaned.replace("[Misleading]", "Another reply:")
    cleaned = cleaned.replace("[Target]",     "Reply to classify:")
    user_content = (
        f"Read the following and classify the stance of the 'Reply to classify'.\n\n"
        f"{cleaned}\n\n"
        f"Stance label (support / deny / query / comment):"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

def evaluate(result):
    golds, preds = result["golds"], result["preds"]
    if len(golds) == 0:
        return {"macro_f1": 0.0, "per_class": [0.0]*4}
    macro_f1  = f1_score(golds, preds, average="macro", zero_division=0)
    per_class = f1_score(golds, preds, average=None, labels=[0,1,2,3], zero_division=0)
    return {"macro_f1": macro_f1, "per_class": per_class.tolist()}

def run_condition(condition, data, predict_fn):
    golds, preds  = [], []
    invalid_count = 0
    skipped_count = 0
    for i, d in enumerate(data):
        if d[condition] is None:
            skipped_count += 1
            continue
        pred = predict_fn(d[condition])
        if pred == "invalid":
            invalid_count += 1
            continue
        golds.append(LABEL_MAP[d["label"]])
        preds.append(LABEL_MAP[pred])
        if (i+1) % 100 == 0:
            print(f"  [{condition}] {i+1}/{len(data)} | 유효 {len(golds)} | invalid {invalid_count}")
    return {"golds": golds, "preds": preds, "invalid_count": invalid_count, "skipped_count": skipped_count, "n": len(golds)}

def make_predict_fn(model, tokenizer):
    def predict(text, max_new_tokens=10):
        messages   = build_prompt(text)
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids  = tokenizer(text_input, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False, temperature=1.0, pad_token_id=tokenizer.eos_token_id)
        new_tokens = output_ids[0][input_ids.shape[-1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        for label in VALID_LABELS:
            if label in raw_output:
                return label
        return "invalid"
    return predict

# 3B adversarial 모델 로드
torch.cuda.empty_cache()
gc.collect()

ft_path   = os.path.expanduser("~/qwen_3b_adv/final")
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer_3b_adv = AutoTokenizer.from_pretrained(ft_path)
base_model_3b    = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
model_3b_adv     = PeftModel.from_pretrained(base_model_3b, ft_path)
model_3b_adv.eval()
print("3B adversarial 모델 로드 완료!")

predict_3b_adv = make_predict_fn(model_3b_adv, tokenizer_3b_adv)

# 평가
all_results_3b_adv = {}
for condition in CONDITIONS:
    print(f"\n{'='*55}")
    print(f"  조건: {condition}")
    print(f"{'='*55}")
    result  = run_condition(condition, dataset, predict_fn=predict_3b_adv)
    metrics = evaluate(result)
    all_results_3b_adv[condition] = {"raw": result, "metrics": metrics}
    pc = metrics["per_class"]
    print(f"  완료 → n={result['n']} | invalid={result['invalid_count']}")
    print(f"  macro-F1 : {metrics['macro_f1']:.4f}")
    print(f"  support={pc[0]:.3f}  deny={pc[1]:.3f}  query={pc[2]:.3f}  comment={pc[3]:.3f}")

# 저장
save_data = {}
for condition, data in all_results_3b_adv.items():
    save_data[condition] = {
        "golds":        data["raw"]["golds"],
        "preds":        data["raw"]["preds"],
        "invalid_count":data["raw"]["invalid_count"],
        "n":            data["raw"]["n"],
        "macro_f1":     data["metrics"]["macro_f1"],
        "per_class_f1": data["metrics"]["per_class"],
    }
with open('experiment_results_3B_adv.json', 'w', encoding='utf-8') as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)
print("\n저장 완료: experiment_results_3B_adv.json")

dev 샘플 수: 1447


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 237.25it/s]
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


3B adversarial 모델 로드 완료!

  조건: reply_only
  [reply_only] 100/1447 | 유효 100 | invalid 0
  [reply_only] 200/1447 | 유효 200 | invalid 0
  [reply_only] 300/1447 | 유효 300 | invalid 0
  [reply_only] 400/1447 | 유효 400 | invalid 0
  [reply_only] 500/1447 | 유효 500 | invalid 0
  [reply_only] 600/1447 | 유효 600 | invalid 0
  [reply_only] 700/1447 | 유효 700 | invalid 0
  [reply_only] 800/1447 | 유효 800 | invalid 0
  [reply_only] 900/1447 | 유효 900 | invalid 0
  [reply_only] 1000/1447 | 유효 1000 | invalid 0
  [reply_only] 1100/1447 | 유효 1100 | invalid 0
  [reply_only] 1200/1447 | 유효 1200 | invalid 0
  [reply_only] 1300/1447 | 유효 1300 | invalid 0
  [reply_only] 1400/1447 | 유효 1400 | invalid 0
  완료 → n=1447 | invalid=0
  macro-F1 : 0.2915
  support=0.000  deny=0.000  query=0.263  comment=0.903

  조건: useful
  [useful] 100/1447 | 유효 100 | invalid 0
  [useful] 200/1447 | 유효 200 | invalid 0
  [useful] 300/1447 | 유효 300 | invalid 0
  [useful] 400/1447 | 유효 400 | invalid 0
  [useful] 500/1447 | 유효 500 | invali

In [5]:
# 3B zero-shot vs adversarial 비교 테이블
with open('experiment_results_3B_adv.json', encoding='utf-8') as f:
    results_3b_zs = json.load(f)

print("\n" + "="*75)
print(f"{'조건':<15} {'3B zero-shot':>13} {'3B adv ft':>10} {'차이':>8}")
print("="*75)
for condition in CONDITIONS:
    m_zs = results_3b_zs[condition]["macro_f1"]
    m_adv = all_results_3b_adv[condition]["metrics"]["macro_f1"]
    print(f"{condition:<15} {m_zs:>13.3f} {m_adv:>10.3f} {m_adv-m_zs:>+8.3f}")
print("="*75)

# per-class 테이블
print("\n" + "="*75)
print(f"{'조건':<15} {'support':>8} {'deny':>8} {'query':>8} {'comment':>8} {'macro':>8}")
print("="*75)
print("[ 3B zero-shot ]")
for condition in CONDITIONS:
    pc = results_3b_zs[condition]["per_class_f1"]
    mf = results_3b_zs[condition]["macro_f1"]
    print(f"{condition:<15} {pc[0]:>8.3f} {pc[1]:>8.3f} {pc[2]:>8.3f} {pc[3]:>8.3f} {mf:>8.3f}")
print("\n[ 3B adversarial ft ]")
for condition in CONDITIONS:
    pc = all_results_3b_adv[condition]["metrics"]["per_class"]
    mf = all_results_3b_adv[condition]["metrics"]["macro_f1"]
    print(f"{condition:<15} {pc[0]:>8.3f} {pc[1]:>8.3f} {pc[2]:>8.3f} {pc[3]:>8.3f} {mf:>8.3f}")
print("="*75)


조건               3B zero-shot  3B adv ft       차이
reply_only              0.291      0.291   +0.000
useful                  0.401      0.401   +0.000
irrelevant              0.373      0.373   +0.000
conflicting             0.380      0.380   +0.000
mixed                   0.359      0.359   +0.000
lexical                 0.404      0.404   +0.000

조건               support     deny    query  comment    macro
[ 3B zero-shot ]
reply_only         0.000    0.000    0.263    0.903    0.291
useful             0.000    0.048    0.641    0.916    0.401
irrelevant         0.000    0.025    0.554    0.913    0.373
conflicting        0.000    0.056    0.542    0.921    0.380
mixed              0.000    0.029    0.489    0.918    0.359
lexical            0.000    0.025    0.673    0.916    0.404

[ 3B adversarial ft ]
reply_only         0.000    0.000    0.263    0.903    0.291
useful             0.000    0.048    0.641    0.916    0.401
irrelevant         0.000    0.025    0.554    0.913    0.37

In [7]:
# zero-shot json 파일 값 확인
with open('experiment_results_3B_zeroshot.json', encoding='utf-8') as f:
    zs = json.load(f)
    
print("저장된 zero-shot 결과:")
for condition in CONDITIONS:
    print(f"{condition}: {zs[condition]['macro_f1']:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'experiment_results_3B_zeroshot.json'

In [9]:
import json

# zero-shot 결과 로드
with open('/home/ubuntu/rumoureval-2019-training-data/experiment_results_3B_zeroshot.json', encoding='utf-8') as f:
    results_3b_zs = json.load(f)

# adv 결과 로드
with open('/home/ubuntu/rumoureval-2019-training-data/ipynb/experiment_results_3B_adv.json', encoding='utf-8') as f:
    results_3b_adv = json.load(f)

print("=== zero-shot ===")
for condition in CONDITIONS:
    print(f"{condition}: {results_3b_zs[condition]['macro_f1']:.4f}")

print("\n=== adversarial ft ===")
for condition in CONDITIONS:
    print(f"{condition}: {results_3b_adv[condition]['macro_f1']:.4f}")

=== zero-shot ===
reply_only: 0.3448
useful: 0.3772
irrelevant: 0.3952
conflicting: 0.3623
mixed: 0.3360
lexical: 0.3387

=== adversarial ft ===
reply_only: 0.2915
useful: 0.4012
irrelevant: 0.3732
conflicting: 0.3799
mixed: 0.3590
lexical: 0.4036


In [10]:
import json

# 모든 결과 파일 로드
base = '/home/ubuntu/rumoureval-2019-training-data'
ipynb = '/home/ubuntu/rumoureval-2019-training-data/ipynb'

with open(f'{base}/experiment_results_0.5B.json', encoding='utf-8') as f:
    r_05_zs = json.load(f)
with open(f'{base}/experiment_results_3B_zeroshot.json', encoding='utf-8') as f:
    r_3b_zs = json.load(f)
with open(f'{ipynb}/experiment_results_3B_adv.json', encoding='utf-8') as f:
    r_3b_adv = json.load(f)

# 1.5B 결과는 all_results에 있음 (Untitled2에서 가져와야 함)
# 일단 있는 것만으로 테이블 출력

CONDITIONS = ["reply_only", "useful", "irrelevant", "conflicting", "mixed", "lexical"]

print("="*90)
print(f"{'조건':<15} {'0.5B_zs':>8} {'1.5B_zs':>8} {'3B_zs':>8} {'1.5B_adv':>9} {'3B_adv':>8}")
print("="*90)

# 1.5B zero-shot, 1.5B adv 결과 수동 입력 (이전에 나온 결과)
r_15_zs = {
    "reply_only": 0.164, "useful": 0.173, "irrelevant": 0.178,
    "conflicting": 0.165, "mixed": 0.151, "lexical": 0.122
}
r_15_adv = {
    "reply_only": 0.331, "useful": 0.395, "irrelevant": 0.384,
    "conflicting": 0.367, "mixed": 0.365, "lexical": 0.367
}

for condition in CONDITIONS:
    m_05  = r_05_zs[condition]["macro_f1"]
    m_15  = r_15_zs[condition]
    m_3b  = r_3b_zs[condition]["macro_f1"]
    m_15a = r_15_adv[condition]
    m_3ba = r_3b_adv[condition]["macro_f1"]
    print(f"{condition:<15} {m_05:>8.3f} {m_15:>8.3f} {m_3b:>8.3f} {m_15a:>9.3f} {m_3ba:>8.3f}")

print("="*90)

조건               0.5B_zs  1.5B_zs    3B_zs  1.5B_adv   3B_adv
reply_only         0.080    0.164    0.345     0.331    0.291
useful             0.074    0.173    0.377     0.395    0.401
irrelevant         0.070    0.178    0.395     0.384    0.373
conflicting        0.078    0.165    0.362     0.367    0.380
mixed              0.075    0.151    0.336     0.365    0.359
lexical            0.082    0.122    0.339     0.367    0.404


In [11]:
!pip install statsmodels -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
